# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook clones the configured GitHub repository branch by default, installs it in editable mode with the optional Playwright extra, installs a headless Chromium browser for dynamic-page network capture, runs a live test, and then displays trusted or untrusted camera outputs. It does not patch source files from notebook cells.

Coordinate enrichment uses source coordinates first, then candidate metadata/title geocoding, then the optional LLM location-inference fallback. The LLM fallback only infers place-name query variants from stream URLs and metadata; Nominatim supplies coordinates, and verified target bounding boxes still decide whether inferred coordinates are accepted.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all traffic cameras from California
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```

The query is intentionally user-controlled. Phrases such as `traffic cameras`, `weather cameras`, or `public live cameras` should be interpreted as camera-type intent, while place names such as `California`, `Greenville, Texas`, or `London, England` are target geography.


Browser capture is supported as a bounded second-stage extraction path for both blind search and SOURCES.md directory rows. Static extraction runs first; rows with generic dynamic-page evidence can then use Playwright to capture `.m3u8`, JSON feed, MapServer, FeatureServer, ArcGIS, and camera API network requests during real browser rendering.


The CLI run uses application progress events from `camera_discovery.cli run --progress-style events`. The notebook renders those real CLI events as in-place progress bars and saves both the combined CLI log and raw progress-event log without patching repository source code.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil

# Colab/repo bootstrap settings. Override with env vars if needed.
REPO_URL = os.environ.get("CAMERA_DISCOVERY_REPO_URL", "https://github.com/dshipley71/camera-discovery.git")
REPO_BRANCH = os.environ.get("CAMERA_DISCOVERY_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("CAMERA_DISCOVERY_REPO_DIR", "/content/camera-discovery"))

print("Notebook bootstrap")
print("repo url:", REPO_URL)
print("branch:", REPO_BRANCH)
print("repo dir:", REPO_DIR)

# No source files are patched by this notebook. To test changes, push/update the GitHub branch
# selected above and rerun the clone/install cells.


In [ ]:
%cd /content

if REPO_DIR.exists():
    print(f"Removing existing repo directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

src_path = REPO_DIR / "src"
assert (src_path / "camera_discovery").exists(), f"Missing package at {src_path / 'camera_discovery'}"

# Make imports work immediately, even before editable install finishes.
os.environ["PYTHONPATH"] = str(src_path)
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Install the package with the optional Playwright extra. Playwright is not a
# default dependency of the package, but this notebook enables bounded browser
# capture when configured rows need real rendering/network capture.
install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".[playwright]", "--no-build-isolation"]
print("$", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)

# Install Chromium for Playwright. In Colab this is required before dynamic-page
# capture can launch a headless browser. Set CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER=false
# only if the browser is already installed in your runtime.
INSTALL_PLAYWRIGHT_BROWSER = os.environ.get("CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER", "true").strip().lower() in {"1", "true", "yes", "on"}
if INSTALL_PLAYWRIGHT_BROWSER:
    browser_cmd = [sys.executable, "-m", "playwright", "install", "chromium"]
    print("$", " ".join(browser_cmd))
    subprocess.run(browser_cmd, check=True)
else:
    print("Skipping Playwright browser install because CAMERA_DISCOVERY_INSTALL_PLAYWRIGHT_BROWSER=false")

# Verify Playwright imports before the live run. Browser capture remains optional:
# static extraction runs first, and only budgeted dynamic-page decisions launch it.
try:
    from playwright.sync_api import sync_playwright  # type: ignore
    print("Playwright import OK; browser/network capture can run when enabled and routed.")
except Exception as exc:
    raise RuntimeError(f"Playwright import failed after installation: {exc!r}")


In [ ]:
import camera_discovery
print("camera_discovery import OK:", camera_discovery.__file__)

# Load provider secrets from Colab userdata when available.
# Configure these in Colab as needed:
#   OLLAMA_API_KEY
#   OPENAI_API_KEY
#   OPENAI_BASE_URL
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#   AWS_DEFAULT_REGION
try:
    from google.colab import userdata  # type: ignore
except Exception as exc:
    userdata = None
    print("Colab userdata not available:", repr(exc))

if userdata is not None:
    for key in [
        "OLLAMA_API_KEY",
        "OPENAI_API_KEY",
        "OPENAI_BASE_URL",
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
        "AWS_SESSION_TOKEN",
        "AWS_DEFAULT_REGION",
    ]:
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
            print(f"Loaded {key} from Colab userdata")


# Sanity-check the installed CLI module without printing help/usage output.
import camera_discovery.cli as camera_cli
print("camera_discovery.cli import OK:", camera_cli.__file__)


| Profile    | Purpose                    | Behavior                                                                                                                                                                             |
| ---------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `fast`     | Quick review/discovery run | Validation is minimized/disabled; trusted `camera.geojson` should not be produced unless trust requirements are met; useful for `untrusted_camera_candidates.geojson` review output. |
| `balanced` | Middle-ground run          | More validation than Fast, but avoids the most expensive checks. Good default for routine testing.                                                                                   |
| `full`     | Most thorough run          | Runs the deepest validation path available, intended for trusted output when geometry and stream validation pass.                                                                    |

The settings cell below exposes only active source-backed controls plus notebook run controls. Deprecated compatibility knobs such as `MAX_STREAMS` are intentionally not exposed. The source still receives its total candidate cap, but the notebook derives that cap from the visible HLS and image-snapshot budgets instead of exposing a separate total-candidate setting. The LLM location-inference cap and candidate semantic-review cap are also derived from that same visible candidate budget so the notebook attempts to enrich/review every accepted candidate that reaches those source stages.


In [ ]:
RUN_PROFILE = os.environ.get("CAMERA_DISCOVERY_PROFILE", "balanced").strip().lower()
if RUN_PROFILE not in {"fast", "balanced", "full"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_PROFILE={RUN_PROFILE!r}; expected fast, balanced, or full")

# User-controlled query. Edit this directly or set CAMERA_DISCOVERY_QUERY in the environment.
# Camera-type terms such as "traffic cameras" are camera intent, not target geography.
USER_QUERY = os.environ.get("CAMERA_DISCOVERY_QUERY", "Get me all traffic cameras from California")

# Other valid examples:
# USER_QUERY = "Get me all cameras from Greenville, Texas"
# USER_QUERY = "Get me all cameras from London, England and New York, New York"

# Notebook run controls. These are used by the notebook wrapper, not the application config.
OUTPUT_DIR = Path(os.environ.get("CAMERA_DISCOVERY_OUTPUT_DIR", "runs/notebook-live-test"))
CLEAN_OUTPUT_DIR = os.environ.get("CAMERA_DISCOVERY_CLEAN_OUTPUT_DIR", "true").strip().lower() in {"1", "true", "yes", "on"}

# LLM provider/model settings.
# This application requires a real LLM provider. Each application section can use
# a different model. Edit the variables below, or set the matching environment
# variables before running this cell.
LLM_PROVIDER = os.environ.get("CAMERA_DISCOVERY_LLM_PROVIDER", "ollama-cloud").strip()

# Defaults are intentionally independent so users can test different models per stage.
DEFAULT_MAIN_MODEL = "gemma4:31b-cloud"
DEFAULT_TARGET_INTENT_MODEL = "gemma3:12b-cloud"
DEFAULT_TARGET_INTENT_FALLBACK_MODEL = "gemma3:12b-cloud"
DEFAULT_GEOCODER_REFEREE_MODEL = "gemma4:31b-cloud"
DEFAULT_LOCATION_INFERENCE_MODEL = "gemma4:31b-cloud"
DEFAULT_CANDIDATE_REVIEW_MODEL = "gemma3:12b-cloud"

# For example, after confirming availability in your provider account, you can use:
# DEFAULT_TARGET_INTENT_MODEL = "qwen3.5:4b"           # fast extraction
# DEFAULT_GEOCODER_REFEREE_MODEL = "gemma4:31b-cloud" # stronger semantic ranking
# DEFAULT_CANDIDATE_REVIEW_MODEL = "qwen3.5:4b"       # faster batch review

MAIN_MODEL = os.environ.get("CAMERA_DISCOVERY_LLM_MODEL", DEFAULT_MAIN_MODEL).strip()
TARGET_INTENT_MODEL = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_MODEL", DEFAULT_TARGET_INTENT_MODEL).strip()
TARGET_INTENT_FALLBACK_MODEL = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL", DEFAULT_TARGET_INTENT_FALLBACK_MODEL).strip()
TARGET_INTENT_TIMEOUT = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_TIMEOUT", "30").strip()
TARGET_INTENT_ATTEMPTS = os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS", "1").strip()
GEOCODER_REFEREE_MODEL = os.environ.get("CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL", DEFAULT_GEOCODER_REFEREE_MODEL).strip()
LOCATION_INFERENCE_MODEL = os.environ.get("CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL", DEFAULT_LOCATION_INFERENCE_MODEL).strip()
LOCATION_INFERENCE_TIMEOUT = os.environ.get("CAMERA_DISCOVERY_LOCATION_INFERENCE_TIMEOUT", "45").strip()
ENABLE_LLM_LOCATION_INFERENCE = os.environ.get("CAMERA_DISCOVERY_ENABLE_LLM_LOCATION_INFERENCE", "true").strip().lower()
LOCATION_INFERENCE_MIN_CONFIDENCE = os.environ.get("CAMERA_DISCOVERY_LOCATION_INFERENCE_MIN_CONFIDENCE", "0.70").strip()
CANDIDATE_REVIEW_MODEL = os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL", DEFAULT_CANDIDATE_REVIEW_MODEL).strip()
CANDIDATE_REVIEW_TIMEOUT = os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_TIMEOUT", "60").strip()
CANDIDATE_REVIEW_BATCH_SIZE = os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_BATCH_SIZE", "8").strip()

# Discovery/candidate budgets. Keep these aligned with active RunConfig fields.
# The notebook does not expose a separate total-candidate knob; the source
# total cap is derived below from these visible per-type candidate budgets.
MAX_SEARCH_QUERIES = os.environ.get("CAMERA_DISCOVERY_MAX_SEARCH_QUERIES", "10").strip()
MAX_SEARCH_RESULTS_PER_QUERY = os.environ.get("CAMERA_DISCOVERY_MAX_SEARCH_RESULTS_PER_QUERY", "8").strip()
MAX_PAGES = os.environ.get("CAMERA_DISCOVERY_MAX_PAGES", "60").strip()
MAX_HLS_CANDIDATES = os.environ.get("CAMERA_DISCOVERY_MAX_HLS_CANDIDATES", "1000").strip()
MAX_IMAGE_SNAPSHOT_CANDIDATES = os.environ.get("CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES", "0").strip()
MAX_DIRECTORY_PAGES = os.environ.get("CAMERA_DISCOVERY_MAX_DIRECTORY_PAGES", "10").strip()
MAX_STRUCTURED_ENDPOINTS_PER_PAGE = os.environ.get("CAMERA_DISCOVERY_MAX_STRUCTURED_ENDPOINTS_PER_PAGE", "25").strip()

# Browser-capture routing controls. Static extraction still runs first; these
# caps bound the second-stage Playwright capture used for dynamic camera pages.
ENABLE_BROWSER_CAPTURE = os.environ.get("CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE", "true").strip().lower()
BROWSER_CAPTURE_TIMEOUT_MS = os.environ.get("CAMERA_DISCOVERY_BROWSER_CAPTURE_TIMEOUT_MS", "15000").strip()
BROWSER_CAPTURE_MIN_SCORE = os.environ.get("CAMERA_DISCOVERY_BROWSER_CAPTURE_MIN_SCORE", "3").strip()
MAX_BROWSER_CAPTURE_PAGES = os.environ.get("CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES", "20").strip()
MAX_BROWSER_CAPTURE_PAGES_BLIND = os.environ.get("CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES_BLIND", "6").strip()
MAX_BROWSER_CAPTURE_PAGES_DIRECTORY = os.environ.get("CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES_DIRECTORY", "12").strip()
MAX_BROWSER_CAPTURE_PAGES_PER_HOST = os.environ.get("CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES_PER_HOST", "3").strip()
BROWSER_CAPTURE_SETTLE_MS = os.environ.get("CAMERA_DISCOVERY_BROWSER_CAPTURE_SETTLE_MS", "1000").strip()
BROWSER_CAPTURE_SCROLL = os.environ.get("CAMERA_DISCOVERY_BROWSER_CAPTURE_SCROLL", "false").strip().lower()
MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE = os.environ.get("CAMERA_DISCOVERY_MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE", "10").strip()
MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE = os.environ.get("CAMERA_DISCOVERY_MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE", "50").strip()
IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS = os.environ.get("CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS", "2.0").strip()


def _nonnegative_int_setting(name: str, value: str) -> int:
    try:
        parsed = int(value)
    except ValueError as exc:
        raise ValueError(f"{name} must be an integer, got {value!r}") from exc
    if parsed < 0:
        raise ValueError(f"{name} must be non-negative, got {parsed}")
    return parsed


# Source RunConfig still reads a total candidate cap env var. For notebook runs,
# derive that cap from HLS budget + image-snapshot budget. Do not set the
# deprecated CAMERA_DISCOVERY_MAX_STREAMS compatibility cap from the notebook.
_TOTAL_CANDIDATE_CAP_ENV = "CAMERA_DISCOVERY_" + "MAX_TOTAL_" + "CANDIDATES"
_SOURCE_TOTAL_CANDIDATE_CAP = (
    _nonnegative_int_setting("CAMERA_DISCOVERY_MAX_HLS_CANDIDATES", MAX_HLS_CANDIDATES)
    + _nonnegative_int_setting("CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES", MAX_IMAGE_SNAPSHOT_CANDIDATES)
)
# Keep review/inference/geocoding caps aligned with the visible candidate budget.
# These are source-backed limits, but the notebook should not expose them as
# independent knobs because doing so can leave discovered candidates unreviewed
# or without enrichment solely due to stale notebook defaults.
MAX_LLM_LOCATION_INFERENCES = str(_SOURCE_TOTAL_CANDIDATE_CAP)
MAX_CANDIDATE_REVIEWS = str(_SOURCE_TOTAL_CANDIDATE_CAP)
MAX_CANDIDATE_GEOCODES = str(_SOURCE_TOTAL_CANDIDATE_CAP)
MAX_STATE_SCALE_CANDIDATE_GEOCODES = str(_SOURCE_TOTAL_CANDIDATE_CAP)
os.environ.pop("CAMERA_DISCOVERY_MAX_STREAMS", None)

# Preserve independent model choices. Do not normalize all stages to one model.
os.environ["CAMERA_DISCOVERY_LLM_PROVIDER"] = LLM_PROVIDER
os.environ["CAMERA_DISCOVERY_LLM_MODEL"] = MAIN_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_MODEL"] = TARGET_INTENT_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL"] = TARGET_INTENT_FALLBACK_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_TIMEOUT"] = TARGET_INTENT_TIMEOUT
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS"] = TARGET_INTENT_ATTEMPTS
os.environ["CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL"] = GEOCODER_REFEREE_MODEL
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL"] = LOCATION_INFERENCE_MODEL
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_TIMEOUT"] = LOCATION_INFERENCE_TIMEOUT
os.environ["CAMERA_DISCOVERY_ENABLE_LLM_LOCATION_INFERENCE"] = ENABLE_LLM_LOCATION_INFERENCE
os.environ["CAMERA_DISCOVERY_MAX_LLM_LOCATION_INFERENCES"] = MAX_LLM_LOCATION_INFERENCES
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_MIN_CONFIDENCE"] = LOCATION_INFERENCE_MIN_CONFIDENCE
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL"] = CANDIDATE_REVIEW_MODEL
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_TIMEOUT"] = CANDIDATE_REVIEW_TIMEOUT
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_BATCH_SIZE"] = CANDIDATE_REVIEW_BATCH_SIZE
os.environ["CAMERA_DISCOVERY_MAX_CANDIDATE_REVIEWS"] = MAX_CANDIDATE_REVIEWS
os.environ["CAMERA_DISCOVERY_MAX_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
os.environ["CAMERA_DISCOVERY_MAX_SEARCH_RESULTS_PER_QUERY"] = MAX_SEARCH_RESULTS_PER_QUERY
os.environ["CAMERA_DISCOVERY_MAX_PAGES"] = MAX_PAGES
os.environ["CAMERA_DISCOVERY_MAX_HLS_CANDIDATES"] = MAX_HLS_CANDIDATES
os.environ["CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES"] = MAX_IMAGE_SNAPSHOT_CANDIDATES
os.environ[_TOTAL_CANDIDATE_CAP_ENV] = str(_SOURCE_TOTAL_CANDIDATE_CAP)
os.environ["CAMERA_DISCOVERY_MAX_DIRECTORY_PAGES"] = MAX_DIRECTORY_PAGES
os.environ["CAMERA_DISCOVERY_MAX_STRUCTURED_ENDPOINTS_PER_PAGE"] = MAX_STRUCTURED_ENDPOINTS_PER_PAGE
os.environ["CAMERA_DISCOVERY_MAX_CANDIDATE_GEOCODES"] = MAX_CANDIDATE_GEOCODES
os.environ["CAMERA_DISCOVERY_MAX_STATE_SCALE_CANDIDATE_GEOCODES"] = MAX_STATE_SCALE_CANDIDATE_GEOCODES
os.environ["CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS"] = IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS

DISCOVERY_MODE = os.environ.get("CAMERA_DISCOVERY_DISCOVERY_MODE", "both").strip().lower()
if DISCOVERY_MODE not in {"blind", "directory", "both", "direct"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_DISCOVERY_MODE={DISCOVERY_MODE!r}")

SOURCES_FILE = Path(os.environ.get("CAMERA_DISCOVERY_SOURCES_FILE", "SOURCES.md"))
SEED_URLS = [url.strip() for url in os.environ.get("CAMERA_DISCOVERY_SEED_URLS", "").split(",") if url.strip()]

required_secret_hint = {
    "ollama": "OLLAMA_API_KEY is required only when using Ollama Cloud; local Ollama may not need it.",
    "ollama-cloud": "OLLAMA_API_KEY is required.",
    "ollama_cloud": "OLLAMA_API_KEY is required.",
    "openai-compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "openai_compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "bedrock": "AWS credentials and AWS_DEFAULT_REGION are required.",
}.get(LLM_PROVIDER.lower(), "provider-specific credentials are required")

print("profile:", RUN_PROFILE)
print("query:", USER_QUERY)
print("output:", OUTPUT_DIR)
print("clean output dir before run:", CLEAN_OUTPUT_DIR)
print("provider:", LLM_PROVIDER)
print("main model:", MAIN_MODEL)
print("target intent model:", TARGET_INTENT_MODEL)
print("target intent fallback model:", TARGET_INTENT_FALLBACK_MODEL)
print("target intent timeout:", TARGET_INTENT_TIMEOUT)
print("target intent attempts:", TARGET_INTENT_ATTEMPTS)
print("geocoder referee model:", GEOCODER_REFEREE_MODEL)
print("location inference model:", LOCATION_INFERENCE_MODEL)
print("location inference timeout:", LOCATION_INFERENCE_TIMEOUT)
print("enable LLM location inference:", ENABLE_LLM_LOCATION_INFERENCE)
print("max LLM location inferences (derived from HLS + image snapshot candidates):", MAX_LLM_LOCATION_INFERENCES)
print("location inference min confidence:", LOCATION_INFERENCE_MIN_CONFIDENCE)
print("candidate review model:", CANDIDATE_REVIEW_MODEL)
print("candidate review timeout:", CANDIDATE_REVIEW_TIMEOUT)
print("candidate review batch size:", CANDIDATE_REVIEW_BATCH_SIZE)
print("max candidate reviews (derived from HLS + image snapshot candidates):", MAX_CANDIDATE_REVIEWS)
print("max search queries:", MAX_SEARCH_QUERIES)
print("max search results per query:", MAX_SEARCH_RESULTS_PER_QUERY)
print("max pages:", MAX_PAGES)
print("max hls candidates:", MAX_HLS_CANDIDATES)
print("max image snapshot candidates:", MAX_IMAGE_SNAPSHOT_CANDIDATES)
print("max directory pages:", MAX_DIRECTORY_PAGES)
print("max structured endpoints per page:", MAX_STRUCTURED_ENDPOINTS_PER_PAGE)
print("browser capture enabled:", ENABLE_BROWSER_CAPTURE)
print("browser capture timeout ms:", BROWSER_CAPTURE_TIMEOUT_MS)
print("browser capture min score:", BROWSER_CAPTURE_MIN_SCORE)
print("max browser pages total/blind/directory/host:", MAX_BROWSER_CAPTURE_PAGES, MAX_BROWSER_CAPTURE_PAGES_BLIND, MAX_BROWSER_CAPTURE_PAGES_DIRECTORY, MAX_BROWSER_CAPTURE_PAGES_PER_HOST)
print("browser capture settle ms / scroll:", BROWSER_CAPTURE_SETTLE_MS, BROWSER_CAPTURE_SCROLL)
print("browser JSON endpoints per page:", MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE)
print("browser network events logged per page:", MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE)
print("max candidate geocodes (derived from HLS + image snapshot candidates):", MAX_CANDIDATE_GEOCODES)
print("max state-scale candidate geocodes (derived from HLS + image snapshot candidates):", MAX_STATE_SCALE_CANDIDATE_GEOCODES)
print("image snapshot refresh validation delay seconds:", IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS)
print("discovery mode:", DISCOVERY_MODE)
print("sources file:", SOURCES_FILE)
print("seed urls:", len(SEED_URLS))
print("credential hint:", required_secret_hint)

if DISCOVERY_MODE == "direct" and not SEED_URLS:
    raise ValueError("direct discovery mode requires CAMERA_DISCOVERY_SEED_URLS or --seed-url values")

if LLM_PROVIDER.lower() in {"ollama-cloud", "ollama_cloud"} and not os.environ.get("OLLAMA_API_KEY"):
    print("WARNING: OLLAMA_API_KEY is not set; Ollama Cloud requests will fail until configured.")


In [ ]:
%%time
# Notebook-only run/progress helpers.
# These helpers stay in the notebook because they are Colab/IPython display glue,
# not camera-discovery application source code. The data comes from real CLI
# progress events emitted by the running application.
from dataclasses import dataclass, field
import html
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from typing import Any

from IPython.display import HTML, display
from camera_discovery.core.progress_events import PROGRESS_EVENT_PREFIX


@dataclass
class NotebookProgressTask:
    key: str
    label: str
    completed: int = 0
    total: int | None = None
    detail: str = ""
    status: str = "active"

    @property
    def percent(self) -> float:
        if not self.total or self.total <= 0:
            return 0.0
        return max(0.0, min(100.0, (self.completed / self.total) * 100.0))

    def update(
        self,
        *,
        label: str | None = None,
        completed: int | None = None,
        total: int | None = None,
        detail: str | None = None,
        status: str | None = None,
    ) -> None:
        if label is not None:
            self.label = label
        if completed is not None:
            self.completed = max(0, int(completed))
        if total is not None:
            self.total = max(0, int(total))
        if detail is not None:
            self.detail = detail
        if status is not None:
            self.status = status


@dataclass
class NotebookProgressState:
    title: str = "camera-discovery live run"
    root_tasks: dict[str, NotebookProgressTask] = field(default_factory=dict)
    target_tasks: dict[str, dict[str, NotebookProgressTask]] = field(default_factory=dict)
    target_labels: dict[str, str] = field(default_factory=dict)
    active_target_id: str | None = None

    def root_task(self, key: str, label: str) -> NotebookProgressTask:
        task = self.root_tasks.get(key)
        if task is None:
            task = NotebookProgressTask(key=key, label=label)
            self.root_tasks[key] = task
        return task

    def target_task(self, target_id: str, key: str, label: str) -> NotebookProgressTask:
        target_map = self.target_tasks.setdefault(target_id, {})
        task = target_map.get(key)
        if task is None:
            task = NotebookProgressTask(key=key, label=label)
            target_map[key] = task
        return task

    def _event_target_id(self, payload: dict[str, Any]) -> str:
        explicit = payload.get("target_id")
        if explicit:
            self.active_target_id = str(explicit)
            return str(explicit)
        # Some application progress events do not currently include a target id.
        # For single-target notebook runs, keep those events under the only known target.
        known_targets = list(self.target_labels)
        if len(known_targets) == 1:
            return known_targets[0]
        return self.active_target_id or "target"

    def handle(self, event: str, payload: dict[str, Any]) -> None:
        target_id = self._event_target_id(payload)
        target_label = str(payload.get("target_label") or payload.get("label") or target_id)
        if target_label and target_label != "None":
            self.target_labels[target_id] = target_label

        if event == "target_resolution_started":
            total = int(payload.get("total") or 1)
            self.root_task("targets", "Resolving targets").update(
                completed=int(payload.get("completed") or 0),
                total=total,
                detail="asking resolver/geocoder",
            )
            return

        if event == "target_resolution_complete":
            targets = int(payload.get("targets") or payload.get("completed") or 0)
            total = max(1, int(payload.get("total") or targets or 1))
            self.root_task("targets", "Resolving targets").update(
                completed=max(targets, total),
                total=max(targets, total),
                detail=f"{targets} target(s) resolved",
                status="complete",
            )
            return

        if event == "target_discovery_started":
            self.target_task(target_id, "scan", "Scanning source rows").update(
                completed=0,
                total=None,
                detail="starting discovery",
            )
            return

        if event == "source_discovery_parallel_started":
            providers = ", ".join(payload.get("providers") or ["directory", "blind"])
            self.target_task(target_id, "scan", "Scanning source rows").update(detail=f"parallel source discovery: {providers}")
            return

        if event == "source_discovery_parallel_complete":
            self.target_task(target_id, "scan", "Scanning source rows").update(
                detail=f"directory {payload.get('directory_rows', 0)} · blind {payload.get('blind_rows', 0)}"
            )
            return

        if event == "source_rows_loading":
            self.target_task(target_id, "scan", "Scanning source rows").update(detail="loading source rows")
            return

        if event == "source_rows_selected":
            total = int(payload.get("primary_rows") or payload.get("selected_rows") or 0)
            selected = int(payload.get("selected_rows") or total)
            discovered = int(payload.get("discovered_rows") or selected)
            self.target_task(target_id, "scan", "Scanning source rows").update(
                completed=0,
                total=total or None,
                detail=f"{selected} selected · {discovered} discovered",
            )
            return

        if event == "source_row_batch_started":
            rows = int(payload.get("rows") or 0)
            task = self.target_task(target_id, "scan", "Scanning source rows")
            if payload.get("phase") != "primary" and rows:
                task.update(total=(task.total or task.completed or 0) + rows, detail=f"checking promoted asset hosts · {rows} row(s)")
            elif rows and not task.total:
                task.update(total=rows)
            return

        if event == "source_row_processed":
            completed = int(payload.get("processed_rows") or 0)
            total = int(payload.get("rows") or 0)
            detail = (
                f"accepted {payload.get('accepted_total', 0)} · "
                f"HLS {payload.get('hls_count', 0)} · "
                f"images {payload.get('image_snapshot_count', 0)}"
            )
            self.target_task(target_id, "scan", "Scanning source rows").update(
                completed=completed,
                total=total or None,
                detail=detail,
            )
            return

        if event == "source_row_batch_complete":
            completed = int(payload.get("processed_rows") or payload.get("rows") or 0)
            total = max(completed, int(payload.get("rows") or completed or 1))
            self.target_task(target_id, "scan", "Scanning source rows").update(
                completed=completed,
                total=total,
                detail=f"accepted {payload.get('accepted_total', 0)} total",
                status="complete",
            )
            return

        if event == "browser_capture_started":
            task = self.target_task(target_id, "browser", "Browser capture")
            attempted = task.completed + 1
            task.update(
                completed=attempted,
                total=max(task.total or attempted, attempted),
                detail=f"{payload.get('source_provider', 'unknown')} · score {payload.get('score', 0)} · {payload.get('url', '')}",
            )
            return

        if event == "browser_capture_page_complete":
            task = self.target_task(target_id, "browser", "Browser capture")
            detail = (
                f"candidates {payload.get('candidates', 0)} · "
                f"HLS {payload.get('hls_urls', 0)} · "
                f"JSON {payload.get('json_urls', 0)} · "
                f"errors {int(bool(payload.get('error')))}"
            )
            task.update(detail=detail)
            return

        if event == "browser_capture_budget_exhausted":
            self.target_task(target_id, "browser", "Browser capture").update(
                detail=f"budget exhausted: {payload.get('skip_reason', 'unknown')}"
            )
            return

        if event == "browser_capture_complete":
            rows = int(payload.get("rows_selected") or 0)
            pages = int(payload.get("pages_attempted") or 0)
            self.target_task(target_id, "browser", "Browser capture").update(
                completed=pages,
                total=max(rows, pages, 1),
                detail=f"selected {rows} · attempted {pages} · candidates {payload.get('candidates', 0)} · errors {payload.get('errors', 0)}",
                status="complete",
            )
            return

        if event == "candidate_metadata_started":
            raw = int(payload.get("raw") or 0)
            self.target_task(target_id, "metadata", "Applying candidate metadata").update(
                completed=0,
                total=max(raw, 1),
                detail=f"{raw} raw candidate(s)",
            )
            return

        if event == "coordinate_enrichment_started":
            total = int(payload.get("unique") or payload.get("total") or 0)
            already = int(payload.get("already_coordinate_bearing") or 0)
            self.target_task(target_id, "coordinates", "Enriching coordinates").update(
                completed=0,
                total=total or None,
                detail=f"already mapped {already} · metadata 0 · geocoded 0 · LLM 0",
            )
            return

        if event == "coordinate_candidate_processed":
            completed = int(payload.get("processed") or 0)
            total = int(payload.get("total") or 0)
            detail = (
                f"mapped {payload.get('coordinate_bearing', 0)} · "
                f"metadata {payload.get('metadata_enriched', 0)} · "
                f"geocoded {payload.get('geocode_enriched', 0)} · "
                f"LLM {payload.get('llm_location_enriched', 0)}"
            )
            self.target_task(target_id, "coordinates", "Enriching coordinates").update(
                completed=completed,
                total=total or None,
                detail=detail,
            )
            return

        if event == "coordinate_enrichment_complete":
            total = int(payload.get("total") or 0)
            detail = (
                f"mapped {payload.get('coordinate_bearing', 0)} · "
                f"metadata {payload.get('metadata_enriched', 0)} · "
                f"geocoded {payload.get('geocode_enriched', 0)} · "
                f"LLM {payload.get('llm_location_enriched', 0)}"
            )
            self.target_task(target_id, "coordinates", "Enriching coordinates").update(
                completed=total,
                total=max(total, 1),
                detail=detail,
                status="complete",
            )
            return

        if event == "scope_review_started":
            total = int(payload.get("unique") or 1)
            # Candidate metadata is a synchronous pre-step; mark it complete once scope review starts.
            if "metadata" in self.target_tasks.get(target_id, {}):
                meta = self.target_tasks[target_id]["metadata"]
                meta.update(completed=meta.total or meta.completed or 1, total=meta.total or 1, status="complete")
            self.target_task(target_id, "scope", "Scope and semantic review").update(
                completed=0,
                total=max(total, 1),
                detail="reviewing target fit",
            )
            return

        if event == "discovery_complete":
            total = int(payload.get("unique") or 1)
            detail = (
                f"raw {payload.get('raw', 0)} · "
                f"unique {payload.get('unique', 0)} · "
                f"mapped {payload.get('coordinate_bearing', 0)} · "
                f"in scope {payload.get('in_scope', 0)}"
            )
            self.target_task(target_id, "scope", "Scope and semantic review").update(
                completed=max(total, 1),
                total=max(total, 1),
                detail=detail,
                status="complete",
            )
            return

        if event == "validation_started":
            self.root_task("validation", "Validating streams and writing outputs").update(
                completed=int(payload.get("completed") or 0),
                total=int(payload.get("total") or 1),
                detail=str(payload.get("description") or "starting"),
            )
            return

        if event == "validation_complete":
            completed = int(payload.get("completed") or payload.get("total") or 1)
            total = max(completed, int(payload.get("total") or 1))
            self.root_task("validation", "Validating streams and writing outputs").update(
                completed=completed,
                total=total,
                detail=str(payload.get("description") or "complete"),
                status="complete",
            )
            return


def parse_progress_event_line(line: str) -> tuple[str, dict[str, Any]] | None:
    if not line.startswith(PROGRESS_EVENT_PREFIX):
        return None
    try:
        message = json.loads(line[len(PROGRESS_EVENT_PREFIX):])
    except json.JSONDecodeError:
        return None
    event = message.get("event")
    payload = message.get("payload") or {}
    if not isinstance(event, str) or not isinstance(payload, dict):
        return None
    return event, payload


def render_progress_html(state: NotebookProgressState) -> str:
    def render_task(task: NotebookProgressTask) -> str:
        total_text = "working" if task.total is None else f"{task.completed}/{task.total}"
        percent = task.percent if task.total else (100.0 if task.status == "complete" else 8.0)
        status = "done" if task.status == "complete" else "running"
        detail = html.escape(task.detail)
        return f"""
        <div class='cd-task'>
          <div class='cd-row'><span>{html.escape(task.label)}</span><span>{html.escape(total_text)} · {status}</span></div>
          <div class='cd-bar'><div class='cd-fill' style='width:{percent:.1f}%'></div></div>
          <div class='cd-detail'>{detail}</div>
        </div>
        """

    parts = [
        """
        <style>
          .camera-discovery-progress {font-family:system-ui,-apple-system,Segoe UI,sans-serif; border:1px solid #d0d5dd; border-radius:12px; padding:12px; background:#fff; color:#101828; max-width:980px;}
          .camera-discovery-progress .cd-title {font-size:16px; font-weight:700; margin-bottom:8px;}
          .camera-discovery-progress .cd-section {border-top:1px solid #eaecf0; padding-top:8px; margin-top:8px;}
          .camera-discovery-progress .cd-target {font-size:13px; font-weight:700; margin-bottom:6px; color:#344054;}
          .camera-discovery-progress .cd-task {margin:8px 0;}
          .camera-discovery-progress .cd-row {display:flex; justify-content:space-between; gap:12px; font-size:13px; font-weight:600;}
          .camera-discovery-progress .cd-row span:last-child {color:#667085; font-weight:500; white-space:nowrap;}
          .camera-discovery-progress .cd-bar {height:10px; border-radius:999px; overflow:hidden; background:#eaecf0; margin-top:4px;}
          .camera-discovery-progress .cd-fill {height:100%; border-radius:999px; background:#2f6fed; transition:width .18s ease;}
          .camera-discovery-progress .cd-detail {font-size:12px; color:#667085; margin-top:2px;}
        </style>
        """,
        f"<div class='camera-discovery-progress'><div class='cd-title'>{html.escape(state.title)}</div>",
    ]
    for key in ["targets"]:
        if key in state.root_tasks:
            parts.append(render_task(state.root_tasks[key]))
    for target_id, tasks in state.target_tasks.items():
        label = state.target_labels.get(target_id, target_id)
        parts.append(f"<div class='cd-section'><div class='cd-target'>{html.escape(label)}</div>")
        for key in ["scan", "browser", "metadata", "coordinates", "scope"]:
            if key in tasks:
                parts.append(render_task(tasks[key]))
        parts.append("</div>")
    if "validation" in state.root_tasks:
        parts.append("<div class='cd-section'><div class='cd-target'>Validation and outputs</div>")
        parts.append(render_task(state.root_tasks["validation"]))
        parts.append("</div>")
    parts.append("</div>")
    return "\n".join(parts)


class NotebookProgressRenderer:
    def __init__(self, title: str = "camera-discovery live run"):
        self.state = NotebookProgressState(title=title)
        self._handle = None

    def update(self, event: str, payload: dict[str, Any]) -> None:
        self.state.handle(event, payload)
        html_obj = HTML(render_progress_html(self.state))
        if self._handle is None:
            self._handle = display(html_obj, display_id=True)
        else:
            self._handle.update(html_obj)


cmd = [
    sys.executable, "-u", "-m", "camera_discovery.cli", "run", USER_QUERY,
    "--profile", RUN_PROFILE,
    "--output-dir", str(OUTPUT_DIR),
    "--discovery-mode", DISCOVERY_MODE,
    "--sources-file", str(SOURCES_FILE),
    "--progress",
    "--progress-style", "events",
]
for url in SEED_URLS:
    cmd.extend(["--seed-url", url])

# Remove stale run artifacts before each live test unless explicitly disabled.
if CLEAN_OUTPUT_DIR and OUTPUT_DIR.exists():
    resolved_output = OUTPUT_DIR.resolve()
    resolved_repo = REPO_DIR.resolve()
    unsafe_roots = {Path("/").resolve(), Path("/content").resolve(), resolved_repo}
    if resolved_output in unsafe_roots:
        raise RuntimeError(f"Refusing to remove unsafe output directory: {resolved_output}")
    print(f"Removing stale output directory: {resolved_output}")
    shutil.rmtree(resolved_output)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combined_log = OUTPUT_DIR / "notebook_cli_combined.log"
progress_events_log = OUTPUT_DIR / "notebook_progress_events.jsonl"
print("$", " ".join(cmd))
print("Streaming CLI output live; notebook progress bars update from real CLI progress events.")
print("combined log:", combined_log)
print("progress events:", progress_events_log)

run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(src_path)
renderer = NotebookProgressRenderer(title=f"camera-discovery: {USER_QUERY}")
process = subprocess.Popen(
    cmd,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
    env=run_env,
)

assert process.stdout is not None
with combined_log.open("w", encoding="utf-8") as log_fh, progress_events_log.open("w", encoding="utf-8") as event_fh:
    for line in process.stdout:
        log_fh.write(line)
        log_fh.flush()
        parsed = parse_progress_event_line(line)
        if parsed is not None:
            event, payload = parsed
            event_fh.write(line)
            event_fh.flush()
            renderer.update(event, payload)
            continue
        print(line, end="")

returncode = process.wait()
print("\nexit:", returncode)
print("combined CLI log:", combined_log)
print("progress events:", progress_events_log)

if returncode != 0:
    raise RuntimeError("camera-discovery run failed; inspect notebook_cli_combined.log and run artifacts")


In [ ]:
from pathlib import Path
import json

for rel in ["logs/source_policy_summary.json", "logs/candidate_discovery_summary.json", "logs/run_summary.json"]:
    path = OUTPUT_DIR / rel
    print("---", rel, "exists=", path.exists())
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:4000])
        except Exception as exc:
            print("Could not parse JSON:", repr(exc))
            print(path.read_text(encoding="utf-8")[:1000])


In [ ]:
summary_path = OUTPUT_DIR / "logs" / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
run_explanation_path = OUTPUT_DIR / "logs" / "run_explanation.json"
run_explanation = json.loads(run_explanation_path.read_text(encoding="utf-8")) if run_explanation_path.exists() else {}

targets = summary.get("targets", [])
print("targets:", len(targets))
for t in targets:
    print(json.dumps({
        "target_id": t.get("target_id"),
        "target_label": t.get("target_label"),
        "canonical_target": t.get("canonical_target"),
        "geometry_status": t.get("geometry_status"),
        "bbox_verified": t.get("bbox_verified"),
        "trust_policy": t.get("trust_policy"),
    }, indent=2))

candidate_summary = summary.get("candidates", {}) or {}
output_summary = summary.get("outputs", {}) or {}
geojson_metrics = run_explanation.get("geojson_metrics", {}) or {}

unique_value = candidate_summary.get("unique_count")
if unique_value is None and isinstance(candidate_summary.get("unique"), list):
    unique_value = len(candidate_summary.get("unique"))
coord_value = (
    output_summary.get("coordinate_bearing_candidates")
    or geojson_metrics.get("coordinate_bearing_candidates")
    or candidate_summary.get("coordinate_bearing_count")
)
if coord_value is None and isinstance(candidate_summary.get("coordinate_bearing"), list):
    coord_value = len(candidate_summary.get("coordinate_bearing"))

trusted_features = output_summary.get("trusted_geojson_features_written", 0)
untrusted_features = output_summary.get("untrusted_geojson_features_written", 0)
geojson_written = output_summary.get("coordinate_bearing_geojson_features_written")
if geojson_written is None:
    geojson_written = trusted_features + untrusted_features
without_geojson = output_summary.get("coordinate_bearing_without_geojson")
if without_geojson is None and coord_value is not None:
    without_geojson = max(0, int(coord_value) - int(geojson_written))

print(json.dumps({
    "unique_candidates": unique_value,
    "coordinate_bearing_candidates": coord_value,
    "trusted_geojson_features": trusted_features,
    "untrusted_geojson_features": untrusted_features,
    "coordinate_bearing_geojson_features_written": geojson_written,
    "coordinate_bearing_without_geojson": without_geojson,
    "geojson_coverage_matches_coordinate_bearing": (without_geojson == 0 if without_geojson is not None else None),
    "trusted_geojson_created": output_summary.get("trusted_geojson_created"),
    "untrusted_geojson_created": output_summary.get("untrusted_geojson_created"),
}, indent=2))


In [ ]:
from IPython.display import Markdown, display

for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'notebook_camera_map.html',
    'review_artifacts.zip',
    'RUN_EXPLANATION.md',
    'logs/run_explanation.json',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/candidate_coordinate_enrichment.json',
    'logs/structured_endpoint_discovery.jsonl',
    'logs/promoted_asset_host_rows.jsonl',
    'logs/page_discovery_signals.jsonl',
    'logs/browser_capture_decisions.jsonl',
    'logs/browser_capture_results.jsonl',
    'logs/browser_capture_errors.jsonl',
    'logs/browser_capture_summary.json',
    'logs/playwright_network_capture_errors.jsonl',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

explanation_md = OUTPUT_DIR / 'RUN_EXPLANATION.md'
if explanation_md.exists():
    display(Markdown(explanation_md.read_text(encoding='utf-8')))
else:
    print('No RUN_EXPLANATION.md was written. Inspect logs/run_summary.json for raw state.')


browser_summary_path = OUTPUT_DIR / 'logs' / 'browser_capture_summary.json'
if browser_summary_path.exists():
    browser_summary = json.loads(browser_summary_path.read_text(encoding='utf-8'))
    print('\nBrowser capture summary')
    for key in [
        'enabled',
        'backend',
        'rows_considered',
        'rows_selected',
        'pages_attempted',
        'candidates',
        'hls_candidates',
        'image_snapshot_candidates',
        'errors',
        'timeouts',
    ]:
        print(f"  {key}: {browser_summary.get(key)}")
    print('  by_source_provider:', browser_summary.get('by_source_provider', {}))

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


## Camera candidate table

This cell loads `camera_candidates_table.csv` first. That CSV is written from non-rejected review candidates, including candidates that do **not** have latitude/longitude and therefore cannot be mapped. The map artifacts are the authoritative place to audit coordinate-bearing candidates: every coordinate-bearing candidate accepted by the run should be written to either trusted `camera.geojson` or untrusted `untrusted_camera_candidates.geojson` when untrusted review output is enabled.

If the CSV is missing, the cell falls back to the available trusted/untrusted GeoJSON.


In [ ]:
from pathlib import Path
from IPython.display import display

TABLE_PATH = OUTPUT_DIR / "camera_candidates_table.csv"
CAMERA_ROWS = []
GEOJSON_PATH = None

if TABLE_PATH.exists() and TABLE_PATH.stat().st_size > 0:
    print("Selected table:", TABLE_PATH)
    try:
        import pandas as pd
        df = pd.read_csv(TABLE_PATH)
        print("Rows:", len(df))
        display_cols = [
            "name", "target_label", "location_display", "location_text", "camera_type", "camera_id", "latitude", "longitude",
            "stream_url", "source_url", "thumbnail_url", "camera_refresh_rate", "map_refresh_rate_seconds",
            "media_type", "trust_level", "validation_status", "scope_status", "discovery_method", "coordinate_source",
            "review_required",
        ]
        existing_cols = [col for col in display_cols if col in df.columns]
        display(df[existing_cols].head(200))
        CAMERA_ROWS = df.to_dict("records")
        if {"latitude", "longitude"}.issubset(df.columns):
            coordinate_rows = df[df["latitude"].notna() & df["longitude"].notna()]
            print("Coordinate-bearing table rows:", len(coordinate_rows))
        else:
            print("Coordinate-bearing table rows: 0 (latitude/longitude columns missing)")
    except Exception as exc:
        print("Could not display candidate CSV table:", repr(exc))
else:
    print("No camera_candidates_table.csv found; falling back to GeoJSON.")
    from camera_discovery.utils.geojson_viewer import (
        load_camera_map_geojson,
        geojson_features_to_rows,
        write_camera_table_csv,
    )
    geojson, source_name, selected_paths = load_camera_map_geojson(OUTPUT_DIR)
    print("Selected GeoJSON files:", [str(path) for path in selected_paths])
    CAMERA_ROWS = geojson_features_to_rows(geojson)
    if not CAMERA_ROWS:
        print("No trusted or untrusted camera GeoJSON features found yet.")
    else:
        table_csv = write_camera_table_csv(OUTPUT_DIR, CAMERA_ROWS)
        print("Rows:", len(CAMERA_ROWS))
        print("CSV table:", table_csv)
        try:
            import pandas as pd
            df = pd.DataFrame(CAMERA_ROWS)
            display(df.head(200))
        except Exception:
            for row in CAMERA_ROWS[:25]:
                print(row)


## Interactive camera map

The map below merges trusted and untrusted GeoJSON when both are present so all geolocated camera candidates are visible. Marker colors distinguish HLS video, image snapshot, and other/future camera media types. Click a marker to see camera metadata, location, camera type, camera ID, live preview/thumbnail, and snapshot refresh information when available.

In Colab, the notebook uses an `IFrame` plus a direct file link fallback because inline HTML rendering can be blocked by the notebook environment.


In [ ]:
from IPython.display import IFrame, HTML, display
from camera_discovery.utils.geojson_viewer import load_camera_map_geojson, write_embedded_camera_map

MAP_GEOJSON, MAP_SOURCE_NAME, MAP_GEOJSON_PATHS = load_camera_map_geojson(OUTPUT_DIR)
print("Selected GeoJSON files for map:", [str(path) for path in MAP_GEOJSON_PATHS])
print("Map features:", len(MAP_GEOJSON.get("features") or []))

if not MAP_GEOJSON.get("features"):
    print("No GeoJSON features available for map display yet. The table above may still contain non-coordinate candidates.")
else:
    MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, output_name="notebook_camera_map.html")
    print("Notebook map:", MAP_PATH)
    print("Open manually if the iframe is blank:", MAP_PATH.resolve())
    display(IFrame(src=str(MAP_PATH), width="100%", height=720))
    display(HTML(f'<p><a href="{MAP_PATH}" target="_blank">Open camera map in a new tab</a></p>'))
